# scGlue Benchmark for spatial mutliomcis data integration on simulated dataset

Notebook benchmarks spatial mutliomcis data integration using scGlue on simulated dataset.

## Loading

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

import omicverse as ov
from itertools import chain

import anndata as ad
import itertools
import networkx as nx
import pandas as pd
import scanpy as sc
import scglue
import seaborn as sns
from matplotlib import rcParams

import snapatac2 as snap

In [ ]:
import torch
print(torch.cuda.current_device())

## scGlue pipeline

In [ ]:
import os
# Set the directory for the datasets and the output directory
data_dir = 'Original_Simulated_Data'
output_dir = 'Processed_Simulated_Data'
os.makedirs(output_dir, exist_ok=True)

# Loop through each dataset
for i in range(1, 6):
    print(f"Process {data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad data.")
    # Read the RNA dataset and set ground truth
    adata_rna = sc.read_h5ad(f'{data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad')
    adata_rna.obs['ground_truth'] = adata_rna.obs['cell_type']
    adata_rna.var['highly_variable'] = adata_rna.var['highly_variable_features']

    # Read the ATAC dataset and preprocess
    adata_atac = sc.read_h5ad(f'{data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_atac.h5ad')
    sc.pp.filter_genes(adata_atac, min_cells=1)

    # Make variable names unique
    adata_rna.var_names_make_unique()
    adata_atac.var_names_make_unique()

    # Manually split cells into RNAseq and ATACseq cells
    adata_atac.obs.index = adata_atac.obs.index + '_atac'
    adata_rna.obs.index = adata_rna.obs.index + '_rna'

    # Scale RNA data and perform PCA
    sc.pp.scale(adata_rna)
    sc.tl.pca(adata_rna, n_comps=100, svd_solver="auto")

    # Perform LSI on ATAC data
    scglue.data.lsi(adata_atac, n_components=100, n_iter=15)
    if i == 1 or i == 3:
        # Get gene annotation and filter genes
        scglue.data.get_gene_annotation(
            adata_rna, gtf="/data/hulei/STmultiVerse/Reference/gencode.vM34.chr_patch_hapl_scaff.annotation.gtf.gz",
            gtf_by="gene_name"
        )
    else:
        # Get gene annotation and filter genes
        scglue.data.get_gene_annotation(
            adata_rna, gtf="/data/hulei/STmultiVerse/Reference/human_v42/gencode.v42.annotation.gtf",
            gtf_by="gene_name"
        )
    adata_rna.var = adata_rna.var.drop('name', axis=1)
    adata_rna = adata_rna[:, adata_rna.var[~adata_rna.var['chromStart'].isnull()].index.tolist()]

    # Split and process ATAC variable names
    split = adata_atac.var_names.str.split(r"[:-]")
    adata_atac.var["chrom"] = split.map(lambda x: x[0])
    adata_atac.var["chromStart"] = split.map(lambda x: x[1]).astype(int)
    adata_atac.var["chromEnd"] = split.map(lambda x: x[2]).astype(int)

    # Generate guidance graph and check it
    guidance = scglue.genomics.rna_anchored_guidance_graph(adata_rna, adata_atac)
    scglue.graph.check_graph(guidance, [adata_rna, adata_atac])

    # Configure datasets for SCGLUE
    scglue.models.configure_dataset(
        adata_rna, "NB", use_highly_variable=True,
        use_layer="raw", use_rep="X_pca"
    )
    scglue.models.configure_dataset(
        adata_atac, "NB", use_highly_variable=True,
        use_rep="X_lsi"
    )

    # Filter guidance graph to only include highly variable features
    guidance_hvf = guidance.subgraph(chain(
        adata_rna.var.query("highly_variable").index,
        adata_atac.var.query("highly_variable").index
    )).copy()

    # Fit SCGLUE model
    glue = scglue.models.fit_SCGLUE(
        {"rna": adata_rna, "atac": adata_atac}, guidance_hvf,
        fit_kws={"directory": "glue"}
    )

    # Encode data using SCGLUE model
    adata_rna.obsm["X_glue"] = glue.encode_data("rna", adata_rna)
    adata_atac.obsm["X_glue"] = glue.encode_data("atac", adata_atac)

    # Combine embeddings for RNA data
    adata_rna.obsm["X_glue"] = glue.encode_data("rna", adata_rna) + glue.encode_data("atac", adata_atac)

    # Perform clustering using the combined embedding
    ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['X_glue'].shape[1],
                    use_rep='X_glue')
    ov.utils.cluster(adata_rna, use_rep='X_glue', method='leiden', resolution=0.6)
    sc.pl.spatial(adata_rna, color=['cell_type', 'leiden'], spot_size=0.12, wspace=0.4)

    if 'artif_dupl' in adata_rna.var.columns:
        del adata_rna.var['artif_dupl']
    # Save the processed dataset
    adata_rna.write_h5ad(f'{output_dir}/Simulated_Dataset_{i}/scglue_multiomics.h5ad', compression='gzip')

In [ ]:
!pip list